# Lobbyists4America — Project Proposal

---

## 1. Client Overview

**Lobbyists4America** operates in one of the most high-stakes arenas in American democracy — where billions of dollars in lobbying spend compete for congressional attention every year. To win in that environment, lobbyists need more than relationships. They need intelligence.

Lobbyists4America addresses this by transforming a decade of congressional social media activity into a strategic edge. The platform surfaces insights across three critical dimensions — the **key topics** shaping legislative debate, the **members** driving those conversations, and the **relationships** forming between them — giving lobbyists a living map of Congress's priorities and power dynamics.

---

## 2. Problem Statement

Lobbying is a data problem. With 535 members of Congress, hundreds of active legislative issues, and billions of dollars of competing interests, lobbyists who rely on intuition alone are flying blind. The challenge is not access to information — it is the ability to cut through the noise and identify where influence is most likely to land.

Lobbyists4America solves this by mining congressional communication patterns over a ten-year period, surfacing the topics, members, and relationships that matter most — and translating them into actionable lobbying strategy.

---

## 3. Dataset Description

The foundation of this analysis is the **US Political Tweets Dataset**, a decade-long corpus of congressional Twitter activity spanning **2008 to 2017**. The dataset captures how members of Congress used Twitter as a public-facing communications channel, offering an unfiltered window into their priorities, positions, and political relationships over time.

The dataset is distributed as a compressed archive (`US_PoliticalTweets.tar.gz`) containing two files:

| File | Format | Size | Description |
|---|---|---|---|
| `tweets.json` | NDJSON | 1.64 GB | 1,243,370 individual tweet records |
| `users.json` | NDJSON | 0.93 MB | 548 congressional member profiles |

Both files use **NDJSON (Newline-Delimited JSON)** format — each line is a self-contained JSON object. This is the standard format for large-scale social media data exports as it allows line-by-line streaming without loading the entire file into memory.

---

## 4. Data Preparation

The raw files were loaded using Python's `json` and `pandas` libraries. Given the 1.64 GB size of `tweets.json`, only the columns required for analysis were retained at load time — reducing the working dataset to **308 MB**.

### 4.1 Column Selection & Justification

Columns were selected based on their direct relevance to the three intelligence dimensions that drive the Lobbyists4America mission.

#### Tweets Dataset

| Column | Dimension | Justification |
|---|---|---|
| `id` | Members | Unique identifier — links and references every tweet without ambiguity |
| `text` | Key topics | Raw material for topic modeling and keyword analysis |
| `created_at` | Key topics | Unix timestamp — tracks how issues evolve across 2008–2017 |
| `user_id` | Members | Links every tweet to a specific congressperson |
| `retweet_count` | Members | Measures influence and reach |
| `favorite_count` | Members | Secondary engagement signal — distinguishes resonant tweets from noise |

#### Users Dataset

| Column | Dimension | Justification |
|---|---|---|
| `id` | Members | Unique identifier — joins back to `user_id` in the tweets dataset |
| `name` | Members | Full name of the congressperson for reporting and visualization |
| `screen_name` | Members | Twitter handle — primary key for external enrichment join |
| `location` | Relationships | Approximate state — used as fallback before enrichment |
| `followers_count` | Members | Measures the reach and influence of each member |
| `verified` | Members | Confirms account authenticity |

---

## 5. Data Diagnostics

Diagnostic checks were run across both datasets to identify any quality issues before analysis.

### 5.1 Tweets — Diagnostic Results

| Check | Result |
|---|---|
| Total rows | 1,243,370 |
| Missing values | None across all columns |
| Duplicate rows | None |
| Duplicate tweet IDs | None |
| Unique members represented | 545 |
| `created_at` data type | Unix timestamp (integer) — requires conversion to datetime |

The tweets dataset is clean. The only required transformation is converting `created_at` from a Unix timestamp integer to a human-readable datetime format before time-based analysis can be performed.

### 5.2 Users — Diagnostic Results

| Check | Result |
|---|---|
| Total rows | 548 |
| Missing values | `party`, `chamber`, `state` — 100% null |
| Duplicate rows | None |
| Duplicate user IDs | None |

The critical finding is that `party`, `chamber`, and `state` — the three most strategically important fields for lobbyists — are entirely absent from the raw Twitter user data. Twitter does not store political affiliation. These fields must be sourced from an external congressional metadata dataset and joined to the users table via `screen_name`.

---

## 6. Data Enrichment Plan

To recover the missing political metadata, the users dataset will be enriched by joining it against the **@unitedstates project** legislators dataset:

```
https://theunitedstates.io/congress-legislators/legislators-historical.csv
```

This dataset maps congressional member names and Twitter handles to their party affiliation, chamber, and state — providing the political context that the raw Twitter data does not contain.

The join key is `screen_name`, which appears in both the users dataset and the legislators dataset.

---

## 7. Entity Relationship Diagram

The enriched data model consists of three tables connected by two joins:

```
TWEETS ||--o{ USERS : "user_id → id"
USERS  ||--o{ LEGISLATORS : "screen_name → screen_name"
```

| Table | Primary Key | Foreign Key | Role |
|---|---|---|---|
| `tweets` | `id` | `user_id` | One tweet per row — the core analytical unit |
| `users` | `id` | — | One member per row — Twitter profile data |
| `legislators` | `screen_name` | — | One member per row — political metadata |

- `tweets` joins to `users` on `user_id → id` — every tweet is attributed to a member
- `users` joins to `legislators` on `screen_name` — every member gains party, chamber, and state

---

## 8. Project Description

The Lobbyists4America project analyzes a decade of congressional Twitter activity — spanning 2008 to 2017 — to extract strategic intelligence for professional lobbyists operating in the United States. Using a dataset of over 1.2 million tweets from 548 members of Congress, the project surfaces patterns in legislative communication across topics, members, and political relationships. The findings are designed to help lobbyists identify which members are most vocal on specific issues, how party lines shape legislative messaging, and where emerging policy conversations are gaining momentum. Beyond the lobbying community, this analysis would also be of interest to political scientists, journalists, policy researchers, and civic technology organizations seeking to understand how Congress communicates with the public. At its core this project transforms raw social media data into a structured intelligence tool — one that makes the priorities and dynamics of Congress legible to anyone with a stake in influencing or understanding American legislation.

---

## 9. Research Questions

The following questions will guide the initial exploration and analysis of the dataset:

**Question 1 — What are the dominant topics in congressional Twitter activity across the 2008–2017 period?**
Which legislative themes appear most frequently in tweet content, and how do those themes shift over time? This addresses the *key topics* dimension directly.

**Question 2 — Which members of Congress are the most influential voices on Twitter, and does that influence align with party or chamber?**
Using tweet volume, retweet counts, and follower reach, which members command the most attention — and does the answer differ between Republicans and Democrats, or between the House and Senate?

**Question 3 — How do congressional Twitter activity patterns shift in response to major political events?**
Are there measurable spikes in tweet volume or topic focus around key legislative moments — elections, budget debates, major legislation — and do those spikes differ by party?

---

## 10. Hypotheses

The following assumptions will be tested and either confirmed or revised as the analysis develops:

**Hypothesis 1 — Republican and Democrat members tweet about fundamentally different topics.**
Based on the political landscape of 2008–2017, it is assumed that party affiliation is the strongest predictor of topic focus — with Republicans emphasizing fiscal policy, defense, and deregulation, and Democrats emphasizing healthcare, social programs, and climate. The data should either confirm this divide or reveal unexpected areas of overlap.

**Hypothesis 2 — A small number of members generate a disproportionate share of high-engagement tweets.**
Congressional Twitter influence is unlikely to be evenly distributed. It is assumed that the top 10% of members by retweet count account for the majority of total engagement — and that these high-influence members are concentrated in Senate leadership roles rather than the House.

**Hypothesis 3 — Tweet volume and topic focus spike significantly around election years.**
It is assumed that 2008, 2010, 2012, 2014, and 2016 will show measurable increases in tweet volume and politically charged language compared to non-election years — reflecting Congress's heightened public communication during electoral periods.

---

## 11. Analytical Approach

The analysis will begin with the `text` and `created_at` fields as the primary entry points — using keyword frequency and time-series aggregation to establish the baseline topic landscape and activity trends across the full 2008–2017 period. From there the focus will shift to the `user_id` field as the bridge to member-level analysis, joining tweets to the enriched users table to segment activity by `party`, `chamber`, and `state`. To measure influence and reach, `retweet_count` and `favorite_count` will serve as the primary evaluation metrics — allowing members and topics to be ranked not just by volume but by the weight their communication carries. The relationship between party affiliation and topic focus will be explored through comparative frequency analysis, looking for statistically meaningful differences in the language used by Republican versus Democrat members. Finally, time-based aggregation around known political events — elections, major legislation, leadership changes — will be used to test whether external moments produce measurable shifts in congressional communication patterns. Throughout, the goal is not just description but actionable insight — every finding will be framed in terms of what it means for a lobbyist deciding where to focus their effort.

---

## 12. Analytical Objectives

Once the data is cleaned, enriched, and loaded into SQL, the analysis will pursue three objectives aligned to the Lobbyists4America mission:

**Key topics** — identify the legislative themes dominating congressional Twitter activity using keyword frequency and topic modeling across the full 2008–2017 corpus.

**Members** — rank members by tweet volume, engagement, and reach to identify the most influential voices by party, chamber, and state.

**Relationships** — map how activity patterns shift over time and across party lines, surfacing emerging issues and coalitions that represent lobbying opportunities.

---

## 9. Methodology Summary

| Phase | Steps |
|---|---|
| Import | Load NDJSON files, filter to required columns, save as CSV |
| Clean | Convert `created_at` to datetime, handle nulls |
| Enrich | Join users to legislators dataset to recover party, chamber, state |
| Store | Load into SQLite via SQLAlchemy |
| Query | SQL queries across volume, members, party, chamber, time |
| Analyze | Topic modeling, sentiment analysis, trend detection |
| Visualize | Charts, timelines, network maps |
| Interpret | Translate findings into lobbying intelligence |
